# Where should rebalancing effort go?

**An independent historical decision study by Abdulaziz Aldoseri.**

Before a cycle-hire network's morning period, an analyst must decide where a limited number of bicycle moves would be most useful. Some stations usually see more departures than arrivals; others see the reverse. A plan can move only a limited number of bikes and can touch only a limited number of stations. The pattern can change on the day.

This notebook examines that allocation decision using **London Santander Cycles recorded CLASSIC-bike journeys**, with evaluation on **2–22 March 2026**. The model chooses an **integer net transfer into each eligible station**: positive means add bikes, negative means remove bikes. Transfers must sum to zero. One removal and one addition count as **one moved bike**, not two. The visitor chooses the bike-move budget **B**, the maximum number of stations touched **K**, and optional protection against intervention at one station.

For station **s**, morning net outflow is recorded departures minus arrivals, counted independently at each event's actual time in **06:00 inclusive to 10:00 exclusive**. The objective minimizes `sum_s abs(forecast net outflow_s − transfer_s)`. Equal scores prefer fewer bike moves, then fewer touched stations. The available information is the station's prior comparable weekdays under a frozen rolling forecast rule. The actual day's flow is reserved for scoring and the explicitly unattainable perfect-information reference.

**What this does and does not establish:** the score is residual recorded flow imbalance under a hypothetical allocation. It is not inventory, unmet trips, stockouts, rider service, cost savings or dispatch feasibility. Historical stock, dock capacity, vehicle routes and operating constraints are unavailable. An abstract feasible allocation is a candidate for investigation, not a truck dispatch plan.

**Attribution:** Powered by TfL Open Data. Contains OS data © Crown copyright and database rights 2016. Geomni UK Map data © and database rights [2019]. [Official journey source](https://cycling.data.tfl.gov.uk/) · [TfL open-data documentation](https://tfl.gov.uk/info-for/open-data-users/our-open-data) · [TfL Transport Data Service terms](https://tfl.gov.uk/corporate/terms-and-conditions/transport-data-service). These are TfL-specific terms with amendments based on OGL v2.0. Data rights are separate from any code licence; no affiliation or endorsement is implied.


## 1. Open the reproduction files

**Local:** open this notebook inside the extracted project folder. Use a Python notebook kernel with the three pinned packages in `requirements.txt`. Local setup does not install dependencies or access the network. The allocation and preparation code itself uses the Python standard library; pandas is used for readable notebook tables, while NumPy/SciPy support the independent solver tests.

**Optional Google Colab:** upload the notebook, run its setup cells and select the study's reproduction ZIP when prompted. The embedded helper validates all paths, sizes and hashes before extracting or importing project code. Setup then installs the pinned dependencies. No Google Drive mount or raw-data download occurs. Obtain the archive from the study's own link and compare its published checksum when available: a manifest verifies integrity, not publisher identity. Hosted Colab execution has not been verified.

The ZIP contains station/day aggregates and source metadata, not raw journey records or bicycle identifiers. The helper refuses traversal, symlinks, conflicting paths, undeclared files, raw journey CSVs and nonempty extraction directories. Original notebook bytes are strictly verified at extraction; later checks allow edited parameters or saved outputs in the working notebook while continuing to verify code, data and requirements.


In [ ]:
"""Standard-library ZIP validation for the mobility reproduction notebook.

Checksums detect corruption and manifest inconsistency, not publisher identity.
Obtain the archive from the study's own download link and compare its published
archive checksum when available. No archive code is executed by this helper.
"""
from __future__ import annotations

from hashlib import sha256
from io import BytesIO
import json
from pathlib import Path, PurePosixPath
import re
import stat
from zipfile import ZipFile

MAX_ARCHIVE_BYTES = 100 * 1024 * 1024
MAX_TOTAL_BYTES = 256 * 1024 * 1024
MAX_FILE_BYTES = 64 * 1024 * 1024
MAX_FILES = 2000
MANIFEST_NAME = "package_manifest.json"


def safe_name(name: str) -> str:
    if not isinstance(name, str) or not name or "\\" in name or ":" in name or "\x00" in name:
        raise ValueError("Invalid archive path")
    path = PurePosixPath(name)
    if path.is_absolute() or any(part in {"", ".", ".."} for part in name.split("/")):
        raise ValueError("Archive path must be relative without traversal")
    if any(part.casefold() in {"private", "private_audit", ".git", ".env", ".deps"} for part in path.parts):
        raise ValueError("Private or repository-internal material is not allowed in the public package")
    if path.suffix.casefold() in {".xlsx", ".xls"} or "journeydataextract" in path.name.casefold():
        raise ValueError("The public package must not contain raw source workbooks or journey CSVs")
    return path.as_posix()


def manifest_entries(raw: bytes) -> dict:
    if len(raw) > 1024 * 1024:
        raise ValueError("Manifest exceeds size limit")
    value = json.loads(raw.decode("utf-8"))
    if not isinstance(value, dict) or not isinstance(value.get("files"), list):
        raise ValueError("Manifest must contain a files array")
    if not 1 <= len(value["files"]) <= MAX_FILES:
        raise ValueError("Invalid manifest file count")
    records, folded = {}, set()
    for item in value["files"]:
        if not isinstance(item, dict):
            raise ValueError("Invalid manifest record")
        name = safe_name(item.get("path"))
        size, digest = item.get("bytes"), item.get("sha256")
        if name == MANIFEST_NAME or name.casefold() in folded:
            raise ValueError("Duplicate, case-colliding or self-referencing manifest path")
        if isinstance(size, bool) or not isinstance(size, int) or not 0 <= size <= MAX_FILE_BYTES:
            raise ValueError("Invalid manifest size")
        if not isinstance(digest, str) or not re.fullmatch(r"[0-9a-f]{64}", digest):
            raise ValueError("Invalid SHA-256 digest")
        records[name] = {"bytes": size, "sha256": digest}
        folded.add(name.casefold())
    if sum(record["bytes"] for record in records.values()) > MAX_TOTAL_BYTES:
        raise ValueError("Manifest total exceeds size limit")
    return records


def verify_package(directory: str | Path, allow_notebook_edits: bool = False) -> dict:
    """Verify declared files; optional working-notebook edits do not exempt code/data.

    Extraction always uses strict verification. At notebook runtime, Jupyter
    saves outputs and edited parameters into .ipynb files, so those working
    documents may differ while all executable modules/data/requirements remain
    hash-checked. Notebook paths still must exist and be regular bounded files.
    """
    root = Path(directory).resolve()
    manifest = root / MANIFEST_NAME
    if manifest.is_symlink() or not manifest.is_file():
        raise ValueError("Package manifest is missing or is a symlink")
    entries = manifest_entries(manifest.read_bytes())
    for name, item in entries.items():
        target = root / name
        if any(part.is_symlink() for part in [target, *target.parents] if part != root.parent):
            raise ValueError("Symlinks are not permitted in a package")
        if not target.resolve().is_relative_to(root) or not target.is_file():
            raise ValueError("Missing or unsafe package file")
        if allow_notebook_edits and target.suffix.casefold() == ".ipynb":
            if target.stat().st_size > MAX_FILE_BYTES:
                raise ValueError("Working notebook exceeds size limit")
            continue
        if target.stat().st_size != item["bytes"] or sha256(target.read_bytes()).hexdigest() != item["sha256"]:
            raise ValueError("Package checksum or size mismatch: " + name)
    return entries


def safe_extract_package(archive: bytes | str | Path, destination: str | Path) -> Path:
    """Verify all members before writing to a new/empty destination directory."""
    if isinstance(archive, bytes):
        raw = archive
    else:
        path = Path(archive)
        if path.stat().st_size > MAX_ARCHIVE_BYTES:
            raise ValueError("Archive exceeds compressed size limit")
        raw = path.read_bytes()
    if len(raw) > MAX_ARCHIVE_BYTES:
        raise ValueError("Archive exceeds compressed size limit")
    output = Path(destination)
    if output.is_symlink():
        raise ValueError("Destination must not be a symlink")
    output = output.resolve()
    if output.exists() and (not output.is_dir() or any(output.iterdir())):
        raise ValueError("Choose a new or empty extraction directory")
    with ZipFile(BytesIO(raw)) as archive_zip:
        infos = archive_zip.infolist()
        if len(infos) > MAX_FILES + 1:
            raise ValueError("Archive has too many entries")
        names, folded, total = {}, set(), 0
        for info in infos:
            name = safe_name(info.filename.rstrip("/") if info.is_dir() else info.filename)
            if name.casefold() in folded:
                raise ValueError("Duplicate or case-colliding archive member")
            folded.add(name.casefold())
            kind = stat.S_IFMT(info.external_attr >> 16)
            if kind not in {0, stat.S_IFREG, stat.S_IFDIR} or (kind == stat.S_IFDIR and not info.is_dir()):
                raise ValueError("Archive contains a nonregular entry")
            if info.flag_bits & 1:
                raise ValueError("Encrypted archive members are not supported")
            if info.is_dir():
                continue
            if not 0 <= info.file_size <= MAX_FILE_BYTES:
                raise ValueError("Archive member exceeds size limit")
            total += info.file_size
            names[name] = info
        if total > MAX_TOTAL_BYTES:
            raise ValueError("Archive exceeds expanded size limit")
        file_names = {name.casefold() for name in names}
        prefixes = {}
        for name in names:
            parts = PurePosixPath(name).parts
            for end in range(1, len(parts) + 1):
                prefix = "/".join(parts[:end])
                folded_prefix = prefix.casefold()
                if folded_prefix in prefixes and prefixes[folded_prefix] != prefix:
                    raise ValueError("Inconsistent case in archive path components")
                prefixes[folded_prefix] = prefix
                if end < len(parts) and folded_prefix in file_names:
                    raise ValueError("Archive file conflicts with a required directory")
        if MANIFEST_NAME not in names:
            raise ValueError("Archive root must contain package_manifest.json")
        if names[MANIFEST_NAME].file_size > 1024 * 1024:
            raise ValueError("Manifest exceeds size limit")
        manifest = archive_zip.read(names[MANIFEST_NAME])
        entries = manifest_entries(manifest)
        if set(names) != set(entries) | {MANIFEST_NAME}:
            raise ValueError("Every archive file must appear exactly once in the manifest")
        verified = {MANIFEST_NAME: manifest}
        for name, item in entries.items():
            info = names[name]
            if info.file_size != item["bytes"]:
                raise ValueError("Member size disagrees with manifest")
            with archive_zip.open(info) as member:
                contents = member.read(MAX_FILE_BYTES + 1)
            if len(contents) != item["bytes"] or sha256(contents).hexdigest() != item["sha256"]:
                raise ValueError("Member checksum disagrees with manifest: " + name)
            verified[name] = contents
    # The manifest is validated before any data or executable source is written.
    output.mkdir(parents=True, exist_ok=True)
    for name, contents in verified.items():
        target = output / name
        if not target.resolve().is_relative_to(output):
            raise ValueError("Unsafe extraction target")
        target.parent.mkdir(parents=True, exist_ok=True)
        with target.open("xb") as handle:
            handle.write(contents)
    verify_package(output)
    return output


In [ ]:
import importlib.util
import subprocess
import sys

try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except (ImportError, ModuleNotFoundError):
    IN_COLAB = False

PROJECT = Path.cwd().resolve()
if not (PROJECT/'mobility_model.py').is_file():
    extracted = PROJECT/'mobility-rebalancing-reproduction'
    if (extracted/'mobility_model.py').is_file():
        verify_package(extracted, allow_notebook_edits=True)
        PROJECT = extracted
    elif IN_COLAB:
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError('Upload exactly one reproduction ZIP.')
        filename, contents = next(iter(uploaded.items()))
        if not filename.lower().endswith('.zip'):
            raise ValueError('Select the study reproduction ZIP.')
        PROJECT = safe_extract_package(contents, extracted)
    else:
        raise FileNotFoundError('Run this notebook inside the extracted folder containing mobility_model.py.')

if (PROJECT/'package_manifest.json').is_file():
    verify_package(PROJECT, allow_notebook_edits=True)
if IN_COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', str(PROJECT/'requirements.txt')])

sys.path.insert(0, str(PROJECT))
import pandas as pd
from mobility_model import all_policies
from pipeline import guarded_freeze, load_panel, forecast
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string(index=False) if hasattr(value, 'to_string') else value)

freeze = guarded_freeze(PROJECT/'outputs')
index = json.loads((PROJECT/'outputs/public_index.json').read_text(encoding='utf-8'))
stations = index['stations']
ids = [station['id'] for station in stations]
eligible = [station['eligible'] for station in stations]
names = {station['id']: station['name'] for station in stations}
scale = index['meta']['scale']
window = index['meta']['selected_history']
assert ids == sorted(set(ids)) and len(ids) == index['meta']['station_count']
assert sum(eligible) == index['meta']['admitted_stations']
assert scale == 60 and window == freeze['selected_history']
assert len(index['days']) == 21
print(f'{len(ids)} ordinary stations; {sum(eligible)} eligible for learned interventions; {window} prior same-weekday calendar lags.')


## 2. Establish chronology and what is observed

Training is **5 January–15 February**, validation **16 February–1 March**, and test **2–22 March 2026**. Training names and the fixed intervention cohort use primary journeys completed strictly before 16 February 00:00. Admission requires primary morning activity in at least four of the six training weeks. Other ordinary stations remain in network outcome scores with zero interventions and **no forecast**.

One global same-weekday history length is selected from 1, 2, 4 and 6 by validation mean absolute error; ties choose the shorter length. Each later forecast averages its previous N calendar same-weekday lags, at least seven days earlier. Unavailable history before the first eligible observation is skipped without extending the horizon. Later zero records mean zero recorded qualifying activity, not proof that the station was open or had spare bikes. The notebook recomputes every displayed forecast from the aggregate panel below.

The naive source times are interpreted as London local winter time; no explicit provider offset was confirmed. The analysis ends before the clock change. The January–March files were modified in June: this is a **retrospective event-time experiment**, assuming operator access to prior completed observations, not a claim that the public feed was available on the historical planning dates.

Whole journeys involving workshop or `_OLD` endpoints are excluded without merging terminal IDs. The primary screen uses CLASSIC bikes and reported duration <=24 hours. Duration alternatives retain the fixed primary cohort and recorded-history mask. See `DATA_PREPARATION.md` and `data/SOURCE_MANIFEST.json` for exact transformations and source hashes.


In [ ]:
display(pd.DataFrame(index['validation']).sort_values('window'))
panel = load_panel('primary_24h', stations)
for record in index['days']:
    day = record['date']
    values, counts = forecast(panel, day, window, eligible, scale)
    assert record['actual'] == panel[day]['actual']
    assert record['departures'] == panel[day]['departures']
    assert record['arrivals'] == panel[day]['arrivals']
    assert record['forecast_num'] == [value if yes else None for value, yes in zip(values, eligible)]
    assert record['history_count'] == counts
print('All 21 actual-flow vectors, forecast vectors and sample counts match the prepared panel and frozen rolling rule.')


## 3. Choose a day and a planning scenario

Change the four values below and rerun this cell and the following sections. The default, **2 March, B=200 and K=20**, was chosen before test evaluation. B and K are visible hypothetical resource limits; they are not observed TfL fleet or staffing quantities.

- **No action:** leave every station unchanged.
- **Greedy pairing:** take the best immediately beneficial feasible donor/recipient move, with deterministic ties.
- **Optimized allocation:** choose transfers under the same restrictions to minimize the forecast objective; then minimize moves and stations touched.
- **Perfect-information reference:** knows the actual day's flow before planning, under the same cohort, budget and protection. It is unavailable to the planner.

Optional `PROTECTED_STATION_ID` prevents any intervention at that terminal. It represents a hypothetical restriction, not an observed closure. K=0 or K=1 must produce no movement because each moved bike needs both a donor and a recipient. Ample resources do not force useless moves.


In [ ]:
DATE = '2026-03-02'
BIKE_BUDGET = 200
TOUCH_LIMIT = 20
PROTECTED_STATION_ID = None  # A station ID string from stations; None means no restriction.

if type(BIKE_BUDGET) is not int or BIKE_BUDGET not in index['meta']['budget_grid']:
    raise ValueError('Choose a declared bike budget: '+str(index['meta']['budget_grid']))
if type(TOUCH_LIMIT) is not int or TOUCH_LIMIT not in index['meta']['touch_grid']:
    raise ValueError('Choose a declared station-touch limit: '+str(index['meta']['touch_grid']))
selected = next((record for record in index['days'] if record['date'] == DATE), None)
if selected is None:
    raise ValueError('Choose a test date from 2026-03-02 through 2026-03-22.')
if PROTECTED_STATION_ID is not None and PROTECTED_STATION_ID not in ids:
    raise ValueError('Protected station must be an exact string ID in the station register.')
protected = ids.index(PROTECTED_STATION_ID) if PROTECTED_STATION_ID is not None else None
forecast_values = [value if value is not None else 0 for value in selected['forecast_num']]
policies = all_policies(ids, forecast_values, selected['actual'], eligible, BIKE_BUDGET, TOUCH_LIMIT, protected, scale)
assert all(result['feasible'] for result in policies.values())
comparison = pd.DataFrame([{'policy': name, **{key: result[key] for key in ['residual', 'forecast_residual', 'worst_station_residual', 'moves', 'touched', 'unused_budget']}} for name, result in policies.items()])
display(comparison)
opt = policies['optimized']
print(f"Observed network net outflow: {sum(selected['actual'])} bikes. Conserved-transfer lower bound: {opt['net_flow_bound']}.")
print(f"Optimized minus no-action residual: {opt['residual']-policies['no_action']['residual']:+d}; optimized minus greedy: {opt['residual']-policies['greedy']['residual']:+d}. Negative is a better flow-balance score.")
print(f"Planned {opt['moves']} of {BIKE_BUDGET} permitted bike moves across {opt['touched']} of {TOUCH_LIMIT} permitted stations.")
if protected is not None:
    print('Protected terminal:', PROTECTED_STATION_ID, names[PROTECTED_STATION_ID], '(zero intervention in every policy).')
print('Feasible under the stated abstract budgets; actual inventory and dispatch feasibility remain unknown.')


In [ ]:
# Full-network station table; display the 20 largest absolute optimized transfers.
station_rows = pd.DataFrame([{
    'station_id': station['id'], 'station': station['name'], 'eligible': station['eligible'],
    'departures': selected['departures'][i], 'arrivals': selected['arrivals'][i],
    'observed_net_outflow': selected['actual'][i],
    'forecast_net_outflow': None if selected['forecast_num'][i] is None else selected['forecast_num'][i]/scale,
    'history_samples': selected['history_count'][i],
    'optimized_transfer': policies['optimized']['transfer'][i],
    'greedy_transfer': policies['greedy']['transfer'][i],
    'observed_residual_after_optimized': selected['actual'][i]-policies['optimized']['transfer'][i],
} for i, station in enumerate(stations)])
station_rows['absolute_transfer'] = station_rows.optimized_transfer.abs()
station_rows = station_rows.sort_values(['absolute_transfer', 'station_id'], ascending=[False, True]).drop(columns='absolute_transfer')
display(station_rows.head(20))
assert station_rows.optimized_transfer.sum() == 0
assert station_rows.loc[~station_rows.eligible, 'optimized_transfer'].eq(0).all()
print('The full station_rows table retains all stations; a missing forecast is intentional for outside-cohort terminals.')


## 4. Check the complete held-out period, including the least-beneficial day

The following comparison recomputes **every test day at the selected B/K**, with **no protection**, and reconciles the results with the packaged evidence. This reference remains unprotected even when a station is protected in the selected-day experiment above. The uncertainty interval is a descriptive seven-day moving-block resample of only 21 paired dates; it does not establish general performance beyond this short winter period.

The least-beneficial reference date is selected **after evaluation** by the greatest optimized-minus-no-action residual, with the earliest date on ties, at the original default B/K. It can still improve on no action; the label does not imply a worsening day occurred. It is shown for transparency and does not replace the calendar-chosen landing example. A better forecast objective need not give a better observed outcome. Ties with a transparent rule are retained.


In [ ]:
daily = []
for record in index['days']:
    forecasts = [value if value is not None else 0 for value in record['forecast_num']]
    results = all_policies(ids, forecasts, record['actual'], eligible, BIKE_BUDGET, TOUCH_LIMIT, None, scale)
    for policy, result in results.items():
        daily.append({'date': record['date'], 'policy': policy, **{key: result[key] for key in ['residual', 'moves', 'touched', 'worst_station_residual']}})
daily_frame = pd.DataFrame(daily)
recorded = pd.read_csv(PROJECT/'outputs/daily_results.csv')
reference = recorded[(recorded.variant == 'primary_24h') & (recorded.budget == BIKE_BUDGET) & (recorded.touch_limit == TOUCH_LIMIT)]
merged = daily_frame.merge(reference, on=['date', 'policy'], suffixes=('_recomputed', '_recorded'), validate='one_to_one')
assert len(merged) == 21*4
for metric in ['residual', 'moves', 'touched', 'worst_station_residual']:
    assert merged[metric+'_recomputed'].eq(merged[metric+'_recorded']).all()
display(daily_frame.pivot(index='date', columns='policy', values='residual'))
display(pd.DataFrame([row for row in index['summaries'] if row['budget'] == BIKE_BUDGET and row['touch_limit'] == TOUCH_LIMIT]))
display(pd.DataFrame([row for row in index['comparisons'] if row['budget'] == BIKE_BUDGET and row['touch_limit'] == TOUCH_LIMIT]))
print('All 84 unprotected selected-setting policy outcomes match the saved daily results.')

adverse_date = index['meta']['adverse_date']
adverse_rows = recorded[(recorded.variant == 'primary_24h') & (recorded.date == adverse_date) & (recorded.budget == index['meta']['default_budget']) & (recorded.touch_limit == index['meta']['default_touch_limit'])]
print('Post-evaluation least-beneficial reference date:', adverse_date, 'at default B/K, no protection.')
display(adverse_rows[['date', 'policy', 'residual', 'moves', 'touched']])


## 5. Weekday patterns, coverage and duration sensitivity

These frozen comparisons cover the full test period and remain **unprotected**. The alternate duration views recompute the fixed forecasting rule and policies on their corresponding recorded flows; the selected history window, intervention cohort and primary availability mask stay fixed. The unrestricted view is conditional on journeys starting in the supplied January–March corpus. It omits pre-January starts and includes journeys known only after their later completion, so it is a source-limited diagnostic rather than a full-population or as-issued backtest.

Forecast errors concern admitted stations only. Outcome scores include all ordinary stations, including new or sparse terminals with no learned intervention. No zero count or first-observation mask establishes operational station availability.


In [ ]:
display(pd.DataFrame([row for row in index['weekday_results'] if row['budget'] == BIKE_BUDGET and row['touch_limit'] == TOUCH_LIMIT and row['group'] in ['weekday', 'weekend']]))
display(pd.DataFrame([row for row in index['sensitivity_comparisons'] if row['budget'] == BIKE_BUDGET and row['touch_limit'] == TOUCH_LIMIT]))
display(pd.DataFrame(index['forecast_diagnostics']).groupby(['variant', 'window'], as_index=False).agg(total_error_num=('error_num','sum'), station_days=('station_count','sum')).assign(mae=lambda table: table.total_error_num/(scale*table.station_days)))
display(pd.DataFrame(index['coverage']))


## 6. Optional full reproduction

The model pipeline can be reproduced entirely from the included aggregates into a **new** result folder. It refuses to overwrite existing frozen/evaluated evidence. Run validation and evaluation as separate commands:

```sh
python pipeline.py --phase validate --output-dir outputs/reproduced
python pipeline.py --phase evaluate --output-dir outputs/reproduced
python -m unittest discover -s tests -v
```

For source-level reproduction, the next optional cell downloads the **six frozen official CSVs (315.65 MB)** only when explicitly enabled. It verifies every expected size and hash, never overwrites an existing file, and invokes the aggregate-only preparation command in a new folder. No raw data is included in the public reproduction archive. Current upstream files that differ from the recorded hashes require a new source review; they do not silently replace the frozen experiment.


In [ ]:
REBUILD_FROM_RAW = False  # Opt in explicitly; downloads six official files only when True.
if REBUILD_FROM_RAW:
    import hashlib
    import urllib.request
    source_manifest = json.loads((PROJECT/'data/SOURCE_MANIFEST.json').read_text(encoding='utf-8'))
    raw_folder = PROJECT/'local-source-downloads'
    aggregate_folder = PROJECT/'reproduced-aggregates'
    if aggregate_folder.exists():
        raise FileExistsError('Choose a new aggregate output directory; existing files will not be overwritten.')
    raw_folder.mkdir(exist_ok=True)
    for item in source_manifest['raw_files']:
        target = raw_folder/item['filename']
        if not target.exists():
            print('Downloading official source:', item['filename'])
            with urllib.request.urlopen(item['url'], timeout=120) as response, target.open('xb') as output:
                downloaded = 0
                while True:
                    chunk = response.read(1024*1024)
                    if not chunk:
                        break
                    downloaded += len(chunk)
                    if downloaded > item['bytes']:
                        raise ValueError('Source exceeds the frozen byte limit: '+item['filename'])
                    output.write(chunk)
        if target.stat().st_size != item['bytes'] or hashlib.sha256(target.read_bytes()).hexdigest() != item['sha256']:
            raise ValueError('Frozen source size/hash mismatch: '+item['filename'])
    subprocess.check_call([sys.executable, str(PROJECT/'prepare_data.py'), '--source-dir', str(raw_folder), '--output-dir', str(aggregate_folder), '--min-active-days', '1', '--min-active-weeks', '4', '--public-only'])
else:
    print('Raw source download/preparation skipped. The included aggregates are sufficient for the notebook and model reproduction.')


## Interpretation and next evidence needed

This eleven-week winter source window and three-week test can demonstrate the trade-off between concentrating effort, conserving bicycles and limiting station touches. It does not validate all-season performance or establish that any proposed move is operationally possible. Historical inventory and dock availability, within-morning paths, vehicle constraints and costs would be needed for a dispatch decision. Preserve worse days and ties when describing the result.

Local cell execution, aggregate reconciliation and extraction tests are the release verification route. **Hosted Google Colab execution is not claimed.**

Powered by TfL Open Data. Contains OS data © Crown copyright and database rights 2016. Geomni UK Map data © and database rights [2019]. The [TfL Transport Data Service terms](https://tfl.gov.uk/corporate/terms-and-conditions/transport-data-service) apply to the source data. No affiliation or endorsement is implied.
